# Model 2: Dropout Prediction - Logistic Regression Baseline

## Overview
This notebook trains a Logistic Regression baseline to predict HIV care gap dropout using cleaned DHS data.

**Target Variable:** `dropout` (1 = HIV+ and no test in 12 months, 0 = retained in care)

**Class Distribution:** 99.92% retained, 0.08% dropout (26 positive cases)

In [27]:
# Setup and Imports
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import sys
import os

# Add project root to path
# We use absolute path discovery to prevent FileNotFoundError
ROOT_PATH = os.path.abspath(os.path.join(os.getcwd(), ".."))
if ROOT_PATH not in sys.path:
    sys.path.append(ROOT_PATH)

import constants as c

# Import scikit-learn components
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    roc_auc_score, 
    recall_score, 
    precision_score, 
    f1_score,
    confusion_matrix,
    classification_report,
    roc_curve
)

print("Setup complete")
print(f"Data path: {c.DHS_CLEAN}")

Setup complete
Data path: /home/skylar_lorena/DS/Phase5/Phase-5-HIV-Care-Gap-AI/data/processed/individual_features_clean.csv


In [33]:
# Load Cleaned Data
df = pd.read_csv(c.DHS_CLEAN)

print(f"Dataset: {df.shape[0]:,} rows, {df.shape[1]} columns")
print(f"Target distribution:\n{df[c.MODEL2_TARGET].value_counts()}")

Dataset: 32,156 rows, 24 columns
Target distribution:
dropout
0    32130
1       26
Name: count, dtype: int64


## Data Preparation

Before splitting and training, we need to:
1. Separate features (X) and target (y)
2. Identify which columns to use as features
3. Ensure no data leakage

In [45]:
# Separate Features and Target
# Define, Prepare Features and Target
X = df[c.MODEL2_FEATURES].copy()
y = df[c.MODEL2_TARGET].copy()

print(f"Features shape: {X.shape}")
print(f"Features dtypes:\n{X.dtypes.value_counts()}")
print(f"Target shape: {y.shape}")
print(f"Dropout rate: {y.mean():.4%}")

Features shape: (32156, 18)
Features dtypes:
bool       9
int64      3
object     3
float64    3
Name: count, dtype: int64
Target shape: (32156,)
Dropout rate: 0.0809%


In [46]:
# Verify Data Quality (No Imputation Needed)
# Handle Missing Values (if any)
missing_before = X.isnull().sum().sum()
if missing_before > 0:
    # Impute categorical columns with mode
    from sklearn.impute import SimpleImputer
    imputer = SimpleImputer(strategy='most_frequent')
    X = pd.DataFrame(imputer.fit_transform(X), columns=X.columns)
    print(f"Imputed {missing_before} missing values")
else:
    print("No missing values found")

Imputed 27523 missing values


In [47]:
# Encode Categorical Variables to Numeric
from sklearn.preprocessing import LabelEncoder

# Identify categorical columns (object dtype)
categorical_cols = X.select_dtypes(include=["object"]).columns.tolist()
print(f"Categorical columns to encode: {categorical_cols}")

# Apply label encoding to each categorical column
label_encoders = {}
for col in categorical_cols:
    le = LabelEncoder()
    X[col] = le.fit_transform(X[col].astype(str))
    label_encoders[col] = le
    print(f"Encoded {col}: {dict(zip(le.classes_, range(len(le.classes_))))}")

print(f"\nAll features now numeric: {X.dtypes.value_counts()}")

Categorical columns to encode: ['county', 'age_group', 'marital_status', 'distance_to_facility', 'ever_tested_hiv', 'tested_hiv_last_12months', 'num_sexual_partners', 'worked_last_12months', 'currently_in_union', 'edu_Higher', 'edu_No education', 'edu_Primary', 'edu_Secondary', 'wealth_Middle', 'wealth_Poorer', 'wealth_Poorest', 'wealth_Richer', 'wealth_Richest']
Encoded county: {'1': 0, '10': 1, '11': 2, '12': 3, '13': 4, '14': 5, '15': 6, '16': 7, '17': 8, '18': 9, '19': 10, '2': 11, '20': 12, '21': 13, '22': 14, '23': 15, '24': 16, '25': 17, '26': 18, '27': 19, '28': 20, '29': 21, '3': 22, '30': 23, '31': 24, '32': 25, '33': 26, '34': 27, '35': 28, '36': 29, '37': 30, '38': 31, '39': 32, '4': 33, '40': 34, '41': 35, '42': 36, '43': 37, '44': 38, '45': 39, '46': 40, '47': 41, '5': 42, '6': 43, '7': 44, '8': 45, '9': 46}
Encoded age_group: {'15-19': 0, '20-24': 1, '25-29': 2, '30-34': 3, '35-39': 4, '40-44': 5, '45-49': 6}
Encoded marital_status: {'Divorced': 0, 'Living together': 1, 

## Train/Test Split

We split the data 80/20 with random_state=42 for reproducibility. This ensures:
- Training set: 80% of data (25,724 samples)
- Test set: 20% of data (6,432 samples)
- Stratified split preserves class distribution, and is critical due to class imbalance (0.08% dropout rate)

In [43]:
# Split Data
# Split 80/20 with stratification
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

# Display split results
print(f"Training set: {X_train.shape[0]:,} samples ({X_train.shape[0]/len(X):.1%})")
print(f"Test set: {X_test.shape[0]:,} samples ({X_test.shape[0]/len(X):.1%})")
print(f"Features: {X_train.shape[1]}")
print(f"\nTraining dropout rate: {y_train.mean():.4%}")
print(f"Test dropout rate: {y_test.mean():.4%}")

Training set: 25,724 samples (80.0%)
Test set: 6,432 samples (20.0%)
Features: 18

Training dropout rate: 0.0816%
Test dropout rate: 0.0777%


## Model Training: Logistic Regression

Logistic Regression serves as our baseline model because:
1. Simple, interpretable, and fast to train
2. Provides probability outputs for AUC-ROC
3. Establishes performance floor for XGBoost comparison

**Hyperparameters:**
- max_iter=1000 (ensures convergence)
- random_state=42 (reproducibility)
- class_weight='balanced' (handles class imbalance)

In [44]:
# Train Logistic Regression
lr_model = LogisticRegression(
    max_iter=1000, random_state=c.RANDOM_STATE, class_weight="balanced", solver="lbfgs"
)

lr_model.fit(X_train, y_train)
print(f"Training complete ({lr_model.n_iter_[0]} iterations)")

ValueError: could not convert string to float: '25-29'